# 15 - Pricing Config Schema

Every bursar deployment starts with a pricing configuration — a YAML file (or its equivalent Python dict) that defines how credits are charged for model inference, tool usage, search, cache, batch jobs, and subscription plans. This notebook walks through every field in that schema, from the simplest required fields to the most nested policy definitions.

The pricing config is validated at load time by the `PricingConfig` Pydantic model in `bursar.config`. Expression strings are parsed and checked against the known metric variables before any request is priced, so typos fail early rather than silently producing wrong costs.

We'll build the config incrementally — starting with a minimal valid config and adding sections one at a time — so each field's role and constraints are clear.

In [ ]:
from decimal import Decimal
from pprint import pprint
from bursar.config import PricingConfig, load_config_from_dict, ConfigError
from bursar.expr import evaluate_expression
from bursar.engine import PricingEngine
from bursar.metrics import UsageMetrics, ToolCall
from bursar.interface.models import PlanDefinition, TierDefinition, OperationPolicy, FeatureLimit

## Top-level schema

The pricing config has eleven top-level keys, two of which are required:

| Key | Type | Required | Default |
|-----|------|----------|--------|
| `version` | `Literal[1]` | Yes | `1` |
| `models` | `dict[str, str]` | **Yes** | — |
| `tools` | `dict[str, str]` | No | `{"_default": "tool_calls * 0"}` |
| `search` | `str` | No | `None` |
| `cache` | `str` | No | `None` |
| `min_balance` | `Decimal` | No | `0` |
| `signup_bonus` | `int` | No | `50` |
| `fixed` | `dict[str, Decimal]` | No | `{}` |
| `plans` | `dict[str, PlanDefinition]` | No | `None` |
| `tiers` | `dict[str, TierDefinition]` | No | `None` |

Only `models` is strictly required — you must define at least one model pricing expression. Every other key is optional and has sensible defaults.

### 1. `version` — schema version

Must be the literal integer `1`. Currently the only valid version. Will increment if the schema changes in a backwards-incompatible way.

In [ ]:
# A version other than 1 is rejected at load time.
try:
    load_config_from_dict({"version": 2, "models": {"_default": "input_tokens * 1"}})
except ConfigError as e:
    print(f"Rejected: {e}")

## 2. `models` — model pricing expressions

The only required section. A dict mapping model identifiers to expression strings. Each expression is evaluated with the `UsageMetrics` variables as inputs.

**Keys** are model IDs (e.g. `gpt-4o`, `claude-sonnet-4-6`, `_default`). The special key `_default` is a fallback: if a request's model name doesn't match any key and `_default` exists, that expression is used. If neither matches, the engine raises a `ValueError`.

**Values** are expression strings using these available variables:

| Variable | Type | Description |
|----------|------|-------------|
| `input_tokens` | `int` | Prompt token count |
| `output_tokens` | `int` | Completion token count |
| `cache_read_tokens` | `int` | Tokens read from LLM context cache |
| `cache_write_tokens` | `int` | Tokens written to context cache |
| `tool_calls` | `int` | Total tool invocations across all tools |
| `search_queries` | `int` | Number of search queries issued |
| `search_results` | `int` | Number of search results processed |
| `web_search_calls` | `int` | Web search API calls |
| `code_exec_calls` | `int` | Code execution sandbox invocations |

The expression language supports `+`, `-`, `*`, `/`, `//`, `%`, comparisons (`<`, `>`, `<=`, `>=`, `==`, `!=`), boolean operators (`and`, `or`, `not`), and these functions: `min`, `max`, `if`, `tier`, `clamp`, `percentile`, `ceil`, `floor`, `round`. Exponentiation (`**`) is blocked for safety. See [Notebook 02](02_expression_language.ipynb) for the full expression language reference.

In [ ]:
# Minimal valid config: just a _default model expression.
config = load_config_from_dict({
    "models": {"_default": "input_tokens * 10 + output_tokens * 30"}
})
print(f"Valid: version={config.version}, models={len(config.models)}")
pprint(config.models)

In [ ]:
# Full models section with multiple providers and a fallback.
full_models = {
    "models": {
        "gpt-4o": "input_tokens * 5 + output_tokens * 15",
        "claude-sonnet-4-6": "input_tokens * 3 + output_tokens * 10",
        "claude-haiku-4-5": "input_tokens * 1 + output_tokens * 4",
        "_default": "input_tokens * 10 + output_tokens * 30",
    }
}
config = load_config_from_dict(full_models)
print(f"{len(config.models)} models registered")

In [ ]:
# Models section must be non-empty — this fails.
try:
    load_config_from_dict({"models": {}})
except ConfigError as e:
    print(f"Rejected: {e}")

### 3. `tools` — per-tool pricing

Defines additional credit costs per tool invocation. Keys are tool names (matching `ToolCall.name` in `UsageMetrics.tool_calls`). The special key `_default` covers any tool not explicitly listed.

Inside tool expressions, an additional variable is available:
- `this_tool_calls` — counts only calls matching this specific tool key (or, for `_default`, the count of unmatched calls). Use this instead of `tool_calls` for per-tool granularity.

In [ ]:
# Tools section with explicit keys and a fallback.
config = load_config_from_dict({
    "models": {"_default": "input_tokens * 10 + output_tokens * 30"},
    "tools": {
        "code_exec": "this_tool_calls * 50",       # 50 credits per code_exec call
        "web_fetch": "this_tool_calls * 10",        # 10 credits per web_fetch call
        "_default": "this_tool_calls * 0",          # everything else free
    }
})

# When omitted, the default tools config is {"_default": "tool_calls * 0"}
print(f"Default tools config: {PricingConfig.model_fields['tools'].default}")

### 4. `search` — search/RAG pricing

A single expression string for search-augmented generation costs. Uses the standard metric variables: `search_queries`, `search_results`, etc.

When set to `None` (or omitted entirely), the search cost component is zero.

In [ ]:
# Search expression: per-query + per-result, capped at 200.
config = load_config_from_dict({
    "models": {"_default": "input_tokens * 10"},
    "search": "min(search_queries * 10 + search_results * 1, 200)"
})
print(f"Search expression: {config.search}")

# Verify with the engine:
engine = PricingEngine(config)
cost = engine.calculate(UsageMetrics(model="_default", input_tokens=100, search_queries=3, search_results=45))
print(f"Search cost: {cost.search_credits} (3*10 + 45*1 = 75, capped at 200)")

### 5. `cache` — cache discount

A single expression string that computes a cache discount (negative cost, reducing the total). Uses `cache_read_tokens` and `cache_write_tokens` among the standard variables.

The expression evaluates to a negative value (savings) or zero. The final total is clamped to `>= 0`.

In [ ]:
# Cache discount expression: -cache_read_tokens * 0.5 (50% of read cost back), capped at model cost.
config = load_config_from_dict({
    "models": {"_default": "input_tokens * 10 + output_tokens * 30"},
    "cache": "clamp(-cache_read_tokens * 0.5, -(input_tokens * 10 + output_tokens * 30), 0)"
})
print(f"Cache expression: {config.cache}")

### 6. `min_balance` — minimum balance floor

A `Decimal` value (non-negative) representing the minimum credit balance a user must maintain. Enforced in `strict` billing mode. Prevents negative balances from concurrent reservation races. Defaults to `0`.

In [ ]:
config = load_config_from_dict({
    "models": {"_default": "input_tokens * 10"},
    "min_balance": 5000
})
print(f"Min balance: {config.min_balance}")

### 7. `signup_bonus` — new-user credit grant

An integer (>= 0) specifying how many credits a new user receives on signup. Defaults to `50`. Used by `CreditManager` when creating new user records.

In [ ]:
config = load_config_from_dict({
    "models": {"_default": "input_tokens * 10"},
    "signup_bonus": 100
})
print(f"Signup bonus: {config.signup_bonus}")

### 8. `fixed` — flat-rate batch jobs

Defines fixed credit charges for batch or async jobs that don't scale with token counts. Keys are job names, values are `Decimal` credit amounts (must be >= 0).

Fixed costs bypass the model/tool/search/cache calculation entirely. The engine looks up the cost by `UsageMetrics.fixed_job`. If the job name isn't in `fixed`, the engine returns 0 for the fixed component.

Common use cases: roadmap generation, review/quiz sessions, diagnostic assessments.

In [ ]:
config = load_config_from_dict({
    "models": {"_default": "input_tokens * 10"},
    "fixed": {
        "roadmap_gen": 5000,
        "review": 2000,
        "diagnostic": 1500,
    }
})
print(f"Fixed jobs: {dict(config.fixed)}")

# Loading a config with negative fixed cost is rejected.
try:
    load_config_from_dict({
        "models": {"_default": "input_tokens * 10"},
        "fixed": {"negative_job": -100}
    })
except Exception as e:
    print(f"Rejected negative fixed cost: {e}")

---

## Credit tiers (`tiers`)

Credit tiers control the order in which credit buckets are consumed, whether they expire, and overdraft behavior. Each tier is a bucket of credits with a priority. Higher priority (larger number = lower priority) buckets are consumed first.

Field schema for each tier entry:

In [ ]:
print(TierDefinition.model_json_schema())
print()
print("Fields:")
for name, field in TierDefinition.model_fields.items():
    print(f"  {name}: {field.annotation}  (default: {field.default})")

**Tier priority walk**: when deducting credits, bursar walks tiers from lowest priority number (highest priority) to highest priority number. Gifted credits (priority 10) are consumed first, then monthly allowance (priority 20), then purchased credits (priority 30).

**Constraints enforced at config load time:**
- At most one tier may set `allow_overdraft=True`
- At most one tier may set `is_default=True` (new credits without an explicit tier key land here)
- `default_ttl_days` > 0 when set
- An explicit `tiers: {}` is rejected (omit the key entirely for "no tiers")

In [ ]:
# Full tiers section — the standard three-tier setup.
config = load_config_from_dict({
    "models": {"_default": "input_tokens * 10"},
    "tiers": {
        "gifted": {
            "name": "Gifted Credits",
            "priority": 10,       # consumed first
            "expires": True,
            "default_ttl_days": 7,
        },
        "monthly": {
            "name": "Monthly Allowance",
            "priority": 20,       # consumed second
            "expires": True,
            "default_ttl_days": 30,
        },
        "purchased": {
            "name": "Purchased Credits",
            "priority": 30,       # consumed last
            "expires": False,
            "allow_overdraft": True,   # can go negative for this tier
            "is_default": True,        # new credits without specified tier go here
        },
    }
})
print(f"{len(config.tiers)} tiers configured")
for k, t in config.tiers.items():
    print(f"  {k}: priority={t.priority}, expires={t.expires}, overdraft={t.allow_overdraft}, default={t.is_default}")

---

## Subscription plans (`plans`)

Plans define subscription tiers with free monthly allowances, rate overrides, feature flags, and per-operation financial-safety policies. Users without a plan use the constructor defaults from `CreditManager`.

In [ ]:
print("PlanDefinition fields:")
for name, field in PlanDefinition.model_fields.items():
    print(f"  {name}: {field.annotation}  (default: {field.default})")

### Plan field details

| Field | Type | Description |
|-------|------|-------------|
| `id` | `str` | Unique plan identifier (required) |
| `name` | `str` | Human-readable plan name (required) |
| `free_allowance` | `Decimal` | Free credits per billing period, default 0 |
| `default_billing_mode` | `"strict"` / `"overdraft"` | Plan-wide billing mode, default `"strict"` |
| `rate_overrides` | `dict[str, str] \| None` | Per-model expression overrides (optional) |
| `features` | `dict[str, Any] \| None` | Arbitrary feature flags for entitlement checks |
| `feature_limits` | `dict[str, FeatureLimit] \| None` | Per-feature invocation-count limits |
| `per_operation` | `dict[str, OperationPolicy] \| None` | Per-operation-type policy overrides |
| `max_concurrent` | `int \| None` | Plan-wide cap on simultaneously-active billed operations |
| `overdraft_floor` | `Decimal \| None` | Negative balance floor in overdraft mode |
| `allowance_period` | `"calendar_month"` / `"rolling_30d"` / `"anniversary"` | Free-allowance reset window |

**Accepted alias**: `billing_mode` is accepted as a short-form alias for `default_billing_mode` in YAML.

### allowance_period modes

Controls when a user's free allowance resets (see `bursar.allowance` for the full resolver):

| Mode | Reset trigger | Anchor required? | Use case |
|------|---------------|------------------|----------|
| `"calendar_month"` | 1st of each UTC month | No | Simple monthly billing (default) |
| `"rolling_30d"` | Every 30 days from plan assignment | Yes (`plan_assigned_at`) | Fair 30-day windows regardless of signup date |
| `"anniversary"` | Monthly on plan-assignment day-of-month, clamped to month length | Yes (`plan_assigned_at`) | Anniversary-based plans |

When `"rolling_30d"` or `"anniversary"` is configured but no anchor is available (user never assigned a plan), both fall back to `"calendar_month"` — no crash, just non-customized windows.

In [ ]:
# feature_limits example: cap roadmap_gen to 1x per day, review to 10x per month.
config = load_config_from_dict({
    "models": {"_default": "input_tokens * 10"},
    "plans": {
        "seeker": {
            "id": "seeker",
            "name": "Seeker",
            "free_allowance": 5000,
            "feature_limits": {
                "roadmap_gen": {
                    "max_calls": 1,
                    "period": "daily",
                    "action": "deny",      # block when exceeded
                },
                "review": {
                    "max_calls": 10,
                    "period": "monthly",
                    "action": "warn",      # non-blocking advisory
                },
            },
            "features": {
                "max_daily_roadmaps": 1,   # boolean/entitlement check (separate from rate limit)
            },
        },
    },
})
seeker = config.plans["seeker"]
print(f"Plan: {seeker.name}")
for feat_name, limit in seeker.feature_limits.items():
    print(f"  {feat_name}: max {limit.max_calls}x per {limit.period}, action={limit.action}")
print(f"  Feature flags: {seeker.features}")

### OperationPolicy

Per-operation policy overrides for financial safety:

In [ ]:
print("OperationPolicy fields:")
for name, field in OperationPolicy.model_fields.items():
    print(f"  {name}: {field.annotation}  (default: {field.default})")

| Field | Type | Description |
|-------|------|-------------|
| `billing_mode` | `"strict"` / `"overdraft"` | Per-operation billing mode override |
| `max_concurrent` | `int \| None` | Max simultaneous leases for this operation type |
| `overdraft_floor` | `Decimal \| None` | Per-operation overdraft floor |

**Policy resolution order** (most specific wins):
1. Per-call `billing_mode` argument to `reserve()` / `run_billed()`
2. `plan.per_operation[operation_type]`
3. `plan.default_billing_mode`
4. Constructor preset on `CreditManager`

### FeatureLimit

Per-feature invocation-count limits for cadence-based rate limiting:

In [ ]:
print("FeatureLimit fields:")
for name, field in FeatureLimit.model_fields.items():
    print(f"  {name}: {field.annotation}  (default: {field.default})")

| Field | Type | Description |
|-------|------|-------------|
| `max_calls` | `int` | Maximum invocations within one period |
| `period` | `"daily"` / `"weekly"` / `"monthly"` / `"yearly"` | Calendar-aligned window |
| `action` | `"deny"` / `"warn"` / `"notify"` | Action when limit reached |

## How the engine combines cost components

The `PricingEngine.calculate()` method evaluates **five independent cost components** and sums them. Each component comes from a separate config section:

| Component | Config source | Expression variables | Sign |
|-----------|---------------|---------------------|------|
| `model_credits` | `models[model_name]` | Standard metrics (input_tokens, output_tokens, etc.) | Positive (cost) |
| `tool_credits` | `tools[tool_name]` | Standard + `this_tool_calls` | Positive (cost) |
| `search_credits` | `search` (single expression) | Standard | Positive (cost) |
| `cache_savings` | `cache` (single expression) | Standard | Negative (discount) |
| `fixed_credits` | `fixed[job_name]` (lookup, not expression) | N/A — flat Decimal | Positive (cost) |

The engine then produces a `CostBreakdown`:

```
raw_total = model_credits + tool_credits + search_credits + cache_savings + fixed_credits
total = max(0, raw_total)   # never negative
```

Each field is quantized to 4 decimal places with `ROUND_HALF_UP`. The `CostBreakdown.total` is the single source of truth — computed once, not re-derived.

When a config section is omitted (e.g. no `search` key), its component is zero. When a model name doesn't match any key, the engine falls back to `_default` or raises `ValueError`.

## Complete plans example

The following config demonstrates every supported field together — all sections, all sub-fields, all nested structures in one dict.

In [ ]:
# Full config demonstrating every supported field — all sections, all sub-fields.
full_config = {
    "version": 1,
    "models": {
        "claude-sonnet-4-6": "input_tokens * 15 + output_tokens * 45",
        "claude-haiku-4-5": "input_tokens * 3 + output_tokens * 9",
        "_default": "input_tokens * 10 + output_tokens * 30",
    },
    "tools": {
        "code_exec": "this_tool_calls * 50",
        "web_fetch": "this_tool_calls * 10",
        "_default": "tool_calls * 0",
    },
    "search": "clamp(min(search_queries * 5 + search_results * 1, 100), 0, 500)",
    "cache": "clamp(-cache_read_tokens * 0.002, -(input_tokens * 10 + output_tokens * 30), 0)",
    "min_balance": 5000,
    "signup_bonus": 50,
    "fixed": {
        "roadmap_gen": 20,
        "diagnostic": 15,
        "review": 5,
    },
    "tiers": {
        "gifted": {"name": "Gifted Credits", "priority": 10, "expires": True, "default_ttl_days": 365},
        "monthly": {"name": "Monthly Allowance", "priority": 20, "expires": True, "default_ttl_days": 30},
        "purchased": {"name": "Purchased Credits", "priority": 30, "expires": False, "allow_overdraft": True, "is_default": True},
    },
    "plans": {
        "seeker": {
            "id": "seeker", "name": "Seeker", "free_allowance": 5000,
            "default_billing_mode": "strict", "max_concurrent": 1, "overdraft_floor": 0,
            "allowance_period": "calendar_month",
            "rate_overrides": {"claude-sonnet-4-6": "min(input_tokens * 3 + output_tokens * 9, 50)"},
            "per_operation": {
                "review": {"billing_mode": "strict", "max_concurrent": 1, "overdraft_floor": 0},
                "roadmap_gen": {"billing_mode": "strict", "max_concurrent": 1, "overdraft_floor": 0},
            },
            "feature_limits": {"roadmap_gen": {"max_calls": 1, "period": "daily", "action": "deny"}},
            "features": {"max_daily_roadmaps": 1, "max_concurrency": 1},
        },
        "sage": {
            "id": "sage", "name": "Sage", "free_allowance": 50000,
            "default_billing_mode": "strict", "max_concurrent": 3, "overdraft_floor": 0,
            "allowance_period": "calendar_month",
            "per_operation": {
                "review": {"billing_mode": "strict", "max_concurrent": 2},
                "roadmap_gen": {"billing_mode": "strict", "max_concurrent": 2},
            },
            "features": {"max_daily_roadmaps": 10, "max_concurrency": 3},
        },
    },
}

config = load_config_from_dict(full_config)

print("=== Validated config ===")
print(f"Models: {list(config.models.keys())}")
print(f"Tools: {list(config.tools.keys())}")
print(f"Search: {config.search[:40]}..." if config.search else "Search: None")
print(f"Cache: {config.cache[:40]}..." if config.cache else "Cache: None")
print(f"Min balance: {config.min_balance}")
print(f"Signup bonus: {config.signup_bonus}")
print(f"Fixed: {list(config.fixed.keys())}")
print(f"Tiers: {list(config.tiers.keys()) if config.tiers else None}")
print(f"Plans: {list(config.plans.keys()) if config.plans else None}")

# Verify all 5 cost components
engine = PricingEngine(config)
cost = engine.calculate(UsageMetrics(
    model="claude-sonnet-4-6", input_tokens=1000, output_tokens=400,
    tool_calls=[ToolCall(name="code_exec")],
))
print(f"\nCost breakdown:  model={cost.model_credits}  tools={cost.tool_credits}  "
      f"search={cost.search_credits}  cache={cost.cache_savings}  fixed={cost.fixed_credits}  total={cost.total}")

## Putting it together: the complete YAML

Every field documented above in one YAML file:

```yaml
version: 1

models:
  claude-sonnet-4-6: input_tokens * 15 + output_tokens * 45
  claude-haiku-4-5: input_tokens * 3 + output_tokens * 9
  _default: input_tokens * 10 + output_tokens * 30

tools:
  code_exec: this_tool_calls * 50
  web_fetch: this_tool_calls * 10
  _default: tool_calls * 0

search: clamp(min(search_queries * 5 + search_results * 1, 100), 0, 500)

cache: clamp(-cache_read_tokens * 0.002, -(input_tokens * 10 + output_tokens * 30), 0)

min_balance: 5000
signup_bonus: 50

fixed:
  roadmap_gen: 20
  diagnostic: 15
  review: 5

tiers:
  gifted:
    name: Gifted Credits
    priority: 10
    expires: true
    default_ttl_days: 365
  monthly:
    name: Monthly Allowance
    priority: 20
    expires: true
    default_ttl_days: 30
  purchased:
    name: Purchased Credits
    priority: 30
    expires: false
    allow_overdraft: true
    is_default: true

plans:
  seeker:
    id: seeker
    name: Seeker
    free_allowance: 5000
    default_billing_mode: strict
    max_concurrent: 1
    overdraft_floor: 0
    allowance_period: calendar_month
    rate_overrides:
      claude-sonnet-4-6: min(input_tokens * 3 + output_tokens * 9, 50)
    per_operation:
      review:
        billing_mode: strict
        max_concurrent: 1
        overdraft_floor: 0
      roadmap_gen:
        billing_mode: strict
        max_concurrent: 1
        overdraft_floor: 0
    feature_limits:
      roadmap_gen:
        max_calls: 1
        period: daily
        action: deny
    features:
      max_daily_roadmaps: 1
      max_concurrency: 1

  sage:
    id: sage
    name: Sage
    free_allowance: 50000
    default_billing_mode: strict
    max_concurrent: 3
    overdraft_floor: 0
    allowance_period: calendar_month
    per_operation:
      review:
        billing_mode: strict
        max_concurrent: 2
      roadmap_gen:
        billing_mode: strict
        max_concurrent: 2
    features:
      max_daily_roadmaps: 10
      max_concurrency: 3
      access_level: full
```

Load via PyYAML, pass to `PricingEngine.from_dict()` or `load_config_from_dict()`.

In [ ]:
# For reference, the full schema of PricingConfig as JSON Schema:
import json
schema = PricingConfig.model_json_schema()
print(json.dumps(schema, indent=2))